# Multi-Stage Progress Example

This notebook demonstrates complex multi-stage tasks with nested progress bars and different throttling strategies.

In [1]:
from fasthtml.common import *
from starlette.responses import StreamingResponse, JSONResponse
import uuid, time, json
import random

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_sizes, badge_styles, badge_colors
from cjm_fasthtml_daisyui.components.data_input.range_slider import range_dui, range_colors
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_align
from cjm_fasthtml_tailwind.utilities.sizing import w, h, container
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_wrap, gap
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CYBERPUNK)

# Initialize
monitor = ProgressMonitor(keep_history=True)
runner = JobRunner(monitor)

In [4]:
# Complex multi-stage pipeline
def data_pipeline(dataset_size=1000):
    from tqdm import tqdm
    import time
    
    results = {
        "stages": [],
        "metrics": {}
    }
    
    # Stage 1: Data Collection
    print("Starting Stage 1: Data Collection")
    collected_data = []
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source A"):
        time.sleep(0.002)
        collected_data.append(f"source_a_{i}")
    
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source B"):
        time.sleep(0.002)
        collected_data.append(f"source_b_{i}")
    
    results["stages"].append("Data Collection Complete")
    results["metrics"]["collected"] = len(collected_data)
    
    # Stage 2: Data Validation
    print("Starting Stage 2: Data Validation")
    valid_data = []
    for item in tqdm(collected_data, desc="2. Validating Data"):
        time.sleep(0.001)
        if random.random() > 0.1:  # 90% pass validation
            valid_data.append(item)
    
    results["stages"].append("Data Validation Complete")
    results["metrics"]["validated"] = len(valid_data)
    
    # Stage 3: Data Transformation
    print("Starting Stage 3: Data Transformation")
    
    # Sub-stage 3.1: Normalization
    normalized = []
    for item in tqdm(valid_data[:len(valid_data)//2], desc="3.1 Normalizing"):
        time.sleep(0.002)
        normalized.append(f"norm_{item}")
    
    # Sub-stage 3.2: Encoding
    encoded = []
    for item in tqdm(valid_data[len(valid_data)//2:], desc="3.2 Encoding"):
        time.sleep(0.002)
        encoded.append(f"enc_{item}")
    
    # Sub-stage 3.3: Merging
    merged = []
    for i in tqdm(range(min(len(normalized), len(encoded))), desc="3.3 Merging"):
        time.sleep(0.001)
        merged.append((normalized[i], encoded[i]))
    
    results["stages"].append("Data Transformation Complete")
    results["metrics"]["transformed"] = len(merged)
    
    # Stage 4: Analysis
    print("Starting Stage 4: Analysis")
    
    # Parallel analysis tasks
    for i in tqdm(range(100), desc="4.1 Statistical Analysis"):
        time.sleep(0.01)
    
    for i in tqdm(range(80), desc="4.2 Pattern Detection"):
        time.sleep(0.012)
    
    for i in tqdm(range(60), desc="4.3 Anomaly Detection"):
        time.sleep(0.015)
    
    results["stages"].append("Analysis Complete")
    results["metrics"]["patterns_found"] = random.randint(10, 50)
    results["metrics"]["anomalies"] = random.randint(1, 10)
    
    # Stage 5: Report Generation
    print("Starting Stage 5: Report Generation")
    
    for i in tqdm(range(50), desc="5.1 Generating Charts"):
        time.sleep(0.02)
    
    for i in tqdm(range(30), desc="5.2 Writing Summary"):
        time.sleep(0.03)
    
    for i in tqdm(range(20), desc="5.3 Finalizing Report"):
        time.sleep(0.04)
    
    results["stages"].append("Report Generation Complete")
    results["report"] = "pipeline_report.pdf"
    
    return results

# Nested loop example
def nested_processing():
    from tqdm import tqdm
    import time
    
    results = []
    
    # Outer loop - batches
    batches = 5
    items_per_batch = 20
    
    for batch in tqdm(range(batches), desc="Processing Batches"):
        batch_results = []
        
        # Inner loop - items in batch
        for item in tqdm(range(items_per_batch), desc=f"Batch {batch+1} Items", leave=False):
            time.sleep(0.05)
            batch_results.append(f"batch_{batch}_item_{item}")
        
        results.append(batch_results)
        time.sleep(0.1)  # Pause between batches
    
    return results

In [5]:
# UI for multi-stage progress
@rt("/")
def home():
    return create_test_page(
        "Multi-Stage Progress Demo",
        Div(
            H1("Multi-Stage Pipeline Progress", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Pipeline controls
            Div(
                H2("Pipeline Configuration", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Label("Dataset Size:", cls=str(label)),
                    Input(
                        id="dataset-size",
                        type="range",
                        min="100",
                        max="2000",
                        value="1000",
                        cls=combine_classes(range_dui, range_colors.primary)
                    ),
                    Span("1000", id="size-display", cls=combine_classes(badge, badge_sizes.lg, m.l(2))),
                    cls=combine_classes(m.b(4))
                ),
                Div(
                    Button("Start Pipeline", id="start-pipeline", cls=combine_classes(btn, btn_colors.primary, m.r(2))),
                    Button("Run Nested Task", id="start-nested", cls=combine_classes(btn, btn_colors.secondary)),
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Stage progress visualization
            Div(
                H2("Pipeline Stages", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    # Stage indicators
                    Div(
                        id="stage-indicators",
                        cls=combine_classes(flex_display, flex_wrap, gap(2), m.b(4))
                    ),
                    
                    # Overall progress
                    Div(
                        P("Overall Pipeline Progress:", cls=str(font_weight.bold)),
                        Progress(id="overall", max="100", value="0", cls=combine_classes(progress, progress_colors.primary, w.full, h(6))),
                        P("0%", id="overall-text", cls=combine_classes(text_align.center, m.t(1))),
                        cls=str(m.b(6))
                    ),
                    
                    # Active progress bars
                    Div(
                        H3("Active Tasks:", cls=combine_classes(font_weight.semibold, m.b(3))),
                        Div(id="active-bars", cls=str(space.y(3))),
                    ),
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Results and metrics
            Div(
                H2("Results", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Pre("No results yet", id="results", cls=combine_classes(bg_dui.base_300, p(4), rounded())),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        ),
        Script("""
            let currentSSE = null;
            let stages = ['Collection', 'Validation', 'Transformation', 'Analysis', 'Report'];
            let currentStage = -1;
            
            // Update dataset size display
            document.getElementById('dataset-size').oninput = function() {
                document.getElementById('size-display').textContent = this.value;
            };
            
            // Initialize stage indicators
            function initStages() {
                const container = document.getElementById('stage-indicators');
                container.innerHTML = '';
                stages.forEach((stage, i) => {
                    const badge = document.createElement('div');
                    badge.className = '""" + combine_classes(badge, badge_sizes.lg, badge_styles.outline) + """';
                    badge.id = `stage-${i}`;
                    badge.textContent = stage;
                    container.appendChild(badge);
                });
            }
            
            // Update stage indicator
            function updateStage(description) {
                // Detect stage from description
                let stageNum = -1;
                if (description.includes('1.')) stageNum = 0;
                else if (description.includes('2.')) stageNum = 1;
                else if (description.includes('3.')) stageNum = 2;
                else if (description.includes('4.')) stageNum = 3;
                else if (description.includes('5.')) stageNum = 4;
                
                if (stageNum > currentStage) {
                    // Mark previous as complete
                    if (currentStage >= 0) {
                        const prev = document.getElementById(`stage-${currentStage}`);
                        if (prev) {
                            prev.className = '""" + combine_classes(badge, badge_sizes.lg, badge_colors.success) + """';
                        }
                    }
                    
                    // Mark current as active
                    currentStage = stageNum;
                    const current = document.getElementById(`stage-${currentStage}`);
                    if (current) {
                        current.className = '""" + combine_classes(badge, badge_sizes.lg, badge_colors.warning) + """';
                    }
                }
            }
            
            // Start pipeline
            document.getElementById('start-pipeline').onclick = async () => {
                if (currentSSE) currentSSE.close();
                
                initStages();
                currentStage = -1;
                document.getElementById('results').textContent = 'Pipeline running...';
                document.getElementById('start-pipeline').disabled = true;
                
                const size = document.getElementById('dataset-size').value;
                
                const resp = await fetch('/api/start-pipeline', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({dataset_size: parseInt(size)})
                });
                
                const data = await resp.json();
                connectSSE(data.job_id);
            };
            
            // Start nested task
            document.getElementById('start-nested').onclick = async () => {
                if (currentSSE) currentSSE.close();
                
                document.getElementById('results').textContent = 'Nested task running...';
                document.getElementById('start-nested').disabled = true;
                
                const resp = await fetch('/api/start-nested', {method: 'POST'});
                const data = await resp.json();
                connectSSE(data.job_id);
            };
            
            // Connect SSE
            function connectSSE(jobId) {
                currentSSE = new EventSource(`/api/stream/${jobId}`);
                
                currentSSE.onmessage = (e) => {
                    const data = JSON.parse(e.data);
                    
                    // Update overall
                    document.getElementById('overall').value = data.progress || 0;
                    document.getElementById('overall-text').textContent = `${(data.progress || 0).toFixed(1)}%`;
                    
                    // Update active bars
                    const barsContainer = document.getElementById('active-bars');
                    barsContainer.innerHTML = '';
                    
                    if (data.bars) {
                        // Sort bars by description for consistent ordering
                        const sortedBars = Object.entries(data.bars).sort((a, b) => 
                            (a[1].desc || '').localeCompare(b[1].desc || ''));
                        
                        sortedBars.forEach(([barId, bar]) => {
                            // Update stage indicator
                            if (bar.desc) {
                                updateStage(bar.desc);
                            }
                            
                            // Create progress bar
                            const barDiv = document.createElement('div');
                            barDiv.className = '""" + combine_classes(p(3), bg_dui.base_300, rounded()) + """';
                            
                            // Determine bar color based on description
                            let barClass = '""" + combine_classes(progress, progress_colors.accent, w.full) + """';
                            if (bar.desc && bar.desc.includes('1.')) barClass = '""" + combine_classes(progress, progress_colors.info, w.full) + """';
                            else if (bar.desc && bar.desc.includes('2.')) barClass = '""" + combine_classes(progress, progress_colors.success, w.full) + """';
                            else if (bar.desc && bar.desc.includes('3.')) barClass = '""" + combine_classes(progress, progress_colors.warning, w.full) + """';
                            else if (bar.desc && bar.desc.includes('4.')) barClass = '""" + combine_classes(progress, progress_colors.error, w.full) + """';
                            else if (bar.desc && bar.desc.includes('5.')) barClass = '""" + combine_classes(progress, progress_colors.primary, w.full) + """';
                            
                            barDiv.innerHTML = `
                                <p class=\"""" + combine_classes(font_size.sm, font_weight.semibold, m.b(1)) + """\">${bar.desc || 'Progress'}</p>
                                <progress class="${barClass}" value="${bar.pct}" max="100"></progress>
                                <p class=\"""" + combine_classes(font_size.xs, m.t(1)) + """\">${bar.pct.toFixed(1)}% (${bar.cur}/${bar.tot})</p>
                            `;
                            barsContainer.appendChild(barDiv);
                        });
                    }
                    
                    // Handle completion
                    if (data.completed) {
                        // Mark last stage as complete
                        if (currentStage >= 0) {
                            const last = document.getElementById(`stage-${currentStage}`);
                            if (last) last.className = '""" + combine_classes(badge, badge_sizes.lg, badge_colors.success) + """';
                        }
                        
                        document.getElementById('start-pipeline').disabled = false;
                        document.getElementById('start-nested').disabled = false;
                        currentSSE.close();
                        
                        // Fetch results
                        fetchResults(jobId);
                    }
                };
                
                currentSSE.onerror = () => {
                    document.getElementById('start-pipeline').disabled = false;
                    document.getElementById('start-nested').disabled = false;
                    currentSSE.close();
                };
            }
            
            // Fetch results
            async function fetchResults(jobId) {
                const resp = await fetch(`/api/results/${jobId}`);
                if (resp.ok) {
                    const results = await resp.json();
                    document.getElementById('results').textContent = JSON.stringify(results, null, 2);
                }
            }
            
            // Initialize
            initStages();
        """)
    )

In [6]:
# Storage for results
job_results = {}

# API endpoints
@rt("/api/start-pipeline", methods=["POST"])
async def start_pipeline(request):
    data = await request.json()
    dataset_size = data.get('dataset_size', 1000)
    
    job_id = str(uuid.uuid4())
    
    def wrapper():
        result = data_pipeline(dataset_size)
        job_results[job_id] = result
    
    runner.start(
        job_id,
        wrapper,
        patch_kwargs={
            "min_delta_pct": 1,  # Update every 1% for smooth progress
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    return JSONResponse({"job_id": job_id})

@rt("/api/start-nested", methods=["POST"])
def start_nested():
    job_id = str(uuid.uuid4())
    
    def wrapper():
        result = nested_processing()
        job_results[job_id] = {"nested_results": result, "total_items": sum(len(b) for b in result)}
    
    runner.start(
        job_id,
        wrapper,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.1,
            "emit_initial": True
        }
    )
    
    return JSONResponse({"job_id": job_id})

@rt("/api/stream/{job_id}")
def stream_multi_stage(job_id: str):
    gen = sse_stream(
        monitor,
        job_id,
        interval=0.05,  # Faster updates for smoother visualization
        heartbeat=15.0,
        wait_for_start=True,
        start_timeout=10.0
    )
    return StreamingResponse(gen, media_type="text/event-stream")

@rt("/api/results/{job_id}")
def get_results(job_id: str):
    if job_id in job_results:
        return JSONResponse(job_results[job_id])
    return JSONResponse({"error": "Results not available"}, status_code=404)

In [7]:
# Start server
server = start_test_server(app)
display(HTMX())

In [8]:
# Stop server when done
server.stop()